In [1]:
%pip install -q librosa numpy torch torchvision scikit-learn

import os
import json
import numpy as np 
import librosa # For audio processing
import librosa.display
import pandas as pd
import torch
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
import torch.nn as nn
from torchvision import models
from sklearn.model_selection import train_test_split

from pathlib import Path
from typing import Tuple, List, Dict
import pickle # For saving and loading data

Note: you may need to restart the kernel to use updated packages.


In [ ]:
def load_annotations(csv_path):
    """
    CSV must have columns: start_sec, end_sec, label (seconds).
    """
    df = pd.read_csv(csv_path)
    # Rename here if your columns differ:
    # df = df.rename(columns={'onset': 'start_sec', 'duration': 'end_sec', ...})
    return df

def make_mel(
    y,
    sr,
    n_fft=1024,
    hop_length=256,
    n_mels=128,
    fmin=20,
    fmax=None
):
    S = librosa.feature.melspectrogram(
        y=y,
        sr=sr,
        n_fft=n_fft,
        hop_length=hop_length,
        n_mels=n_mels,
        fmin=fmin,
        fmax=fmax
    )
    S_db = librosa.power_to_db(S, ref=np.max)
    return S_db  # shape: (n_mels, time_frames)

def segment_to_mel(y, sr, start_sec, end_sec, **mel_kwargs):
    start_sample = int(start_sec * sr)
    end_sample = int(end_sec * sr)
    # Clip to valid range
    start_sample = max(0, start_sample)
    end_sample = min(len(y), end_sample)
    segment = y[start_sample:end_sample]

    if len(segment) == 0:
        return None  # or handle specially

    mel = make_mel(segment, sr, **mel_kwargs)
    return mel

def normalize_mel(mel):
    """
    Normalize to [0, 1] or standardize to mean/std. 
    Here: simple [0,1] per-spectrogram.
    """
    m_min = mel.min()
    m_max = mel.max()
    if m_max == m_min:
        return np.zeros_like(mel)
    return (mel - m_min) / (m_max - m_min)

def preprocess_audio_segments(
    audio_path,
    csv_path,
    out_dir,
    sr=16000,
    n_fft=1024,
    hop_length=256,
    n_mels=128,
    fmin=20,
    fmax=None,
    min_duration_sec=1.0
):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    
    # Load audio once
    y, sr = librosa.load(audio_path, sr=sr)  # resample if needed
    
    # Load annotations
    df = load_annotations(csv_path)

    records = []  # to store metadata (path, label, etc.)
    
    for idx, row in df.iterrows():
        start = float(row["start_sec"])
        end = float(row["end_sec"])
        label = row["label"]

        if end - start < min_duration_sec:
            continue  # skip extremely short segments

        mel = segment_to_mel(
            y, sr, start, end,
            n_fft=n_fft,
            hop_length=hop_length,
            n_mels=n_mels,
            fmin=fmin,
            fmax=fmax
        )

        if mel is None:
            continue

        mel = normalize_mel(mel).astype(np.float32)

        # Save each segment as its own .npy
        mel_filename = f"segment_{idx:05d}.npy"
        mel_path = out_dir / mel_filename
        np.save(mel_path, mel)

        records.append({
            "file": str(mel_path),
            "label": label,
            "start_sec": start,
            "end_sec": end
        })

    # Save metadata CSV
    meta_df = pd.DataFrame.from_records(records)
    meta_df.to_csv(out_dir / "metadata.csv", index=False)

    print(f"Saved {len(records)} segments to {out_dir}")